# 10 — OpenBB-driven rolling backtest

Run the rolling-window backtest engine over a configurable date window, then visualize:
- per-day calibration RMSE for both models;
- parameter drift (rBergomi H is the most interesting);
- model risk spread (mean absolute Heston-vs-rBergomi price diff in bps).

Note: free OpenBB providers (yfinance) generally expose only the latest chain. For an actual historical backtest, configure a paid provider (Intrinio / FMP) via OpenBB, or seed `data/cache/` with archived snapshots.

## Context

We use the OpenBB-driven backtest harness to recalibrate both models on each trading day in a chosen window, then track:

- per-day IV-RMSE for each model;
- the parameter time series ($H_t$, $\eta_t$, $\rho_t$ for rBergomi; the five Heston parameters);
- the **model risk spread** — mean absolute price difference between the two calibrated models on a held-out vanilla portfolio, in bps of spot.

On free-tier yfinance the harness operates on SPY; for true historical SPX backtests you need a paid provider (Intrinio, FMP, Polygon). The data loader refuses to cache historical dates against live-only providers.

In [ ]:
import datetime as dt
import matplotlib.pyplot as plt
import pandas as pd

from volengine.backtesting import BacktestConfig, run_backtest

In [ ]:
cfg = BacktestConfig(
    symbol='SPY',
    start=dt.date(2024, 1, 2),
    end=dt.date(2024, 1, 31),
    provider='yfinance',
    rbergomi_n_paths=10_000,  # smaller for notebook demo
)
result = run_backtest(cfg)
df = result.to_dataframe()
df.head()

## RMSE time series

**Figure.** Per-day IV-RMSE for Heston and rBergomi. The smoother / lower line is the better-fitting model on that day. rBergomi typically wins at short maturities; both models track each other on longer-dated quotes.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(df['date'], df['heston_rmse'] * 100, 'o-', color='C0', label='Heston')
axes[0].plot(df['date'], df['rbergomi_rmse'] * 100, 's-', color='C3', label='rBergomi')
axes[0].axhline(1.5, color='gray', ls=':', lw=1, label='1.5 vol-pt target')
axes[0].set_ylabel('in-sample IV RMSE\n(vol points)')
axes[0].set_title(f'{cfg.symbol} rolling backtest: calibration quality, parameter drift, '
                  f'and model risk\n{cfg.start} to {cfg.end} ({cfg.provider})', fontsize=12)
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(df['date'], df['rbergomi_H'], 'o-', color='purple')
axes[1].axhline(0.1, color='gray', ls=':', lw=1, label='SPX-typical H ≈ 0.1')
axes[1].set_ylabel('rBergomi roughness\n$H_t$')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].plot(df['date'], df['model_risk_spread_bps'], 'o-', color='crimson')
axes[2].set_ylabel('model risk spread\n(bps of spot)')
axes[2].set_xlabel('trading date'); axes[2].grid(alpha=0.3)

fig.text(0.5, -0.02,
         'Top: per-day in-sample IV RMSE for each model (lower = better fit). '
         'Middle: the calibrated rBergomi roughness $H_t$ — smooth drift signals '
         'a healthy calibration, jumps usually flag data anomalies. Bottom: the '
         'mean absolute Heston-vs-rBergomi price gap on the day\'s vanilla book, '
         'in bps of spot — the production-relevant model-risk number.',
         ha='center', fontsize=9, wrap=True)
fig.tight_layout()
fig.savefig('../results/figures/backtest_overview.png', dpi=120, bbox_inches='tight')
plt.show()

## Model risk spread

**Figure.** Mean absolute price difference between calibrated Heston and rBergomi on the day's quote universe, in bps of spot. This is the production-relevant punchline: it quantifies how much your P&L would differ between the two models for the same vanilla book.

## Parameter drift

**Figure.** Calibrated parameters versus calendar time. Smooth drift indicates a healthy calibration; jumps usually signal data anomalies (corporate actions, settlement gaps) rather than model failure. Warm-start propagates each day's parameters as the next day's initial guess; cold-start re-runs DE every day for comparison.